## Qualidade & Observabilidade — Squad 2
### Objetivo
Este notebook implementa o monitoramento de qualidade de dados das tabelas Silver e Gold do Squad 2, aplicando as regras definidas na planilha de validação do projeto `merca-data-platform`.
### Escopo
| Tabela | Camada | Regras |

| `ecommerce_categorias` | Silver | CAT-R01 a CAT-R05 |

| `ecommerce_itens_pedido` | Silver | ITP-R01 a ITP-R08 |

| `ecommerce_produtos` | Silver | PRD-R01 a PRD-R06 |

### Resultado esperado
Ao final de cada seção, um gráfico de barras mostra a **quantidade de registros com erro por regra**, agrupados por tipo de problema (Completude, Unicidade, Domínio, Formato, Negócio, Consistência).
### Dependências
- Tabelas Silver gravadas em `squad2/silver/`
- Secret Scope `adls-ecommerce` configurado
- Helpers: `feat_squad2_99_helpers`
### Convenções
| Sigla | Significado |

| `erro_count` | Total de registros que violam a regra |

| `pct_erro` | Percentual em relação ao total do lote |

| `PASS` | Regra sem violações |

| `FAIL` | Regra com ao menos 1 violação |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import logging

log = logging.getLogger(__name__)

# ── Paleta de cores por tipo de problema ──────────────────────────────────────
CORES_TIPO = {
    "Completude"  : "#E74C3C",  # vermelho
    "Unicidade"   : "#E67E22",  # laranja
    "Domínio"     : "#F1C40F",  # amarelo
    "Formato"     : "#3498DB",  # azul
    "Referencial" : "#9B59B6",  # roxo
    "Negócio"     : "#1ABC9C",  # verde-água
    "Consistência": "#95A5A6",  # cinza
}

# ── Função central de gráfico ─────────────────────────────────────────────────
def plot_erros(resultados: list[dict], titulo: str) -> None:
    """
    Gera gráfico de barras horizontais com quantidade de erros por regra.

    Args:
        resultados : lista de dicts com chaves:
                     codigo, descricao, tipo, erro_count, total
        titulo     : nome da tabela para o título do gráfico
    """
    if not resultados:
        print(" Nenhum resultado para plotar.")
        return

    df_plot = pd.DataFrame(resultados)
    df_plot["pct"]    = (df_plot["erro_count"] / df_plot["total"] * 100).round(2)
    df_plot["label"]  = df_plot["codigo"] + " — " + df_plot["descricao"]
    df_plot["cor"]    = df_plot["tipo"].map(CORES_TIPO).fillna("#BDC3C7")
    df_plot           = df_plot.sort_values("erro_count", ascending=True)

    fig, ax = plt.subplots(figsize=(12, max(4, len(df_plot) * 0.7)))

    bars = ax.barh(
        df_plot["label"],
        df_plot["erro_count"],
        color=df_plot["cor"],
        edgecolor="white",
        height=0.6,
    )

    # Anotação de valor + percentual em cada barra
    for bar, pct in zip(bars, df_plot["pct"]):
        width = bar.get_width()
        ax.text(
            width + max(df_plot["erro_count"]) * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{int(width):,}  ({pct}%)",
            va="center",
            fontsize=9,
            color="#2C3E50",
        )

    # Legenda de tipos
    legend_patches = [
        mpatches.Patch(color=cor, label=tipo)
        for tipo, cor in CORES_TIPO.items()
        if tipo in df_plot["tipo"].values
    ]
    ax.legend(
        handles=legend_patches,
        title="Tipo de problema",
        loc="lower right",
        fontsize=8,
    )

    ax.set_title(f"Erros de Qualidade — {titulo}", fontsize=13, fontweight="bold", pad=14)
    ax.set_xlabel("Quantidade de registros com erro", fontsize=10)
    ax.set_xlim(0, df_plot["erro_count"].max() * 1.18)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()
    print(f"\n{'='*60}")
    print(f"  RESUMO — {titulo}")
    print(f"{'='*60}")
    for _, row in df_plot.sort_values("erro_count", ascending=False).iterrows():
        status = "FAIL" if row["erro_count"] > 0 else "PASS"
        print(f"  {status}  {row['codigo']:<10} {row['erro_count']:>6,} erros  ({row['pct']}%)")
    print(f"{'='*60}")

print(" Setup concluído — funções de qualidade carregadas.")

## 1. ecommerce_categorias
### Regras aplicadas
| Código | Tipo | Regra |

| CAT-R01 | Completude | `id_categoria` e `nome_categoria` não podem ser nulos |

| CAT-R02 | Unicidade | `id_categoria` não pode se repetir no lote |

| CAT-R03 | Formato | `nome_categoria` não pode ser string vazia ou só espaços |

| CAT-R04 | Negócio | Número de categorias raiz não pode ser zero |

| CAT-R05 | Consistência | `id_categoria` deve ser numérico positivo |

In [0]:
# ── 1. Leitura da Silver ──────────────────────────────────────────────────────
log.info("Lendo Silver: ecommerce_categorias")

df_cat = ler_delta_silver("ecommerce_categorias")   # função do helpers
total  = df_cat.count()

log.info(f"Total de registros lidos: {total:,}")

# ── 2. Aplicação das regras ───────────────────────────────────────────────────

# CAT-R01 — Completude: nulos em id_categoria ou nome_categoria
r01 = df_cat.filter(
    df_cat["id_categoria"].isNull() | df_cat["nome_categoria"].isNull()
).count()

# CAT-R02 — Unicidade: id_categoria duplicado no lote
from pyspark.sql import functions as F

r02 = (
    df_cat.groupBy("id_categoria")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# CAT-R03 — Formato: nome_categoria vazio ou só espaços
r03 = df_cat.filter(
    F.trim(F.col("nome_categoria")) == ""
).count()

# CAT-R04 — Negócio: categorias raiz = 0 (sem pai definido)
# Assume coluna id_categoria_pai — nulo significa raiz
if "id_categoria_pai" in df_cat.columns:
    raiz_count = df_cat.filter(df_cat["id_categoria_pai"].isNull()).count()
    r04 = 1 if raiz_count == 0 else 0   # 1 erro se não há nenhuma raiz
else:
    r04 = 0
    log.warning("CAT-R04: coluna id_categoria_pai não encontrada — regra ignorada")

# CAT-R05 — Consistência: id_categoria não numérico positivo
r05 = df_cat.filter(
    df_cat["id_categoria"].cast("int").isNull() |
    (df_cat["id_categoria"].cast("int") <= 0)
).count()

# ── 3. Consolidação dos resultados ────────────────────────────────────────────
resultados_cat = [
    {"codigo": "CAT-R01", "descricao": "Nulos em id/nome_categoria",       "tipo": "Completude",   "erro_count": r01, "total": total},
    {"codigo": "CAT-R02", "descricao": "id_categoria duplicado no lote",   "tipo": "Unicidade",    "erro_count": r02, "total": total},
    {"codigo": "CAT-R03", "descricao": "nome_categoria vazio/só espaços",  "tipo": "Formato",      "erro_count": r03, "total": total},
    {"codigo": "CAT-R04", "descricao": "Sem categorias raiz no lote",      "tipo": "Negócio",      "erro_count": r04, "total": total},
    {"codigo": "CAT-R05", "descricao": "id_categoria não numérico/negativo","tipo": "Consistência", "erro_count": r05, "total": total},
]

# ── 4. Gráfico ────────────────────────────────────────────────────────────────
plot_erros(resultados_cat, "ecommerce_categorias")

## 2. ecommerce_itens_pedido
### Regras aplicadas
| Código | Tipo | Regra |
| ITP-R01 | Completude | Nenhuma coluna obrigatória pode ser nula |

| ITP-R03 | Domínio | `quantidade` deve ser > 0 |

| ITP-R04 | Domínio | `preco_unitario` deve ser > 0 |

| ITP-R05 | Unicidade | `id_item_pedido` não pode se repetir no lote |

| ITP-R06 | Negócio | `desconto_aplicado` não pode ser > `preco_unitario` |

 | ITP-R07 | Consistência | `quantidade` deve ser inteiro sem casas decimais |

In [0]:
# ── 1. Leitura da Silver ──────────────────────────────────────────────────────
log.info("Lendo Silver: ecommerce_itens_pedido")

df_itp = ler_delta_silver("ecommerce_itens_pedido")
total  = df_itp.count()

log.info(f"Total de registros lidos: {total:,}")
log.info(f"Colunas disponíveis: {df_itp.columns}")

# ── 2. Colunas obrigatórias esperadas ─────────────────────────────────────────
# Adapta automaticamente ao schema real da tabela
COLUNAS_OBRIGATORIAS_ITP = [
    c for c in [
        "id_item_pedido",
        "id_pedido",
        "quantidade",
        "preco_unitario",
    ]
    if c in df_itp.columns
]

# ── 3. Aplicação das regras ───────────────────────────────────────────────────
from pyspark.sql import functions as F

# ITP-R01 — Completude: nulos em qualquer coluna obrigatória presente
r01 = 0
for col in COLUNAS_OBRIGATORIAS_ITP:
    r01 += df_itp.filter(F.col(col).isNull()).count()

# ITP-R03 — Domínio: quantidade <= 0
r03 = df_itp.filter(F.col("quantidade") <= 0).count() \
    if "quantidade" in df_itp.columns else 0

# ITP-R04 — Domínio: preco_unitario <= 0
r04 = df_itp.filter(F.col("preco_unitario") <= 0).count() \
    if "preco_unitario" in df_itp.columns else 0

# ITP-R05 — Unicidade: id_item_pedido duplicado no lote
r05 = (
    df_itp.groupBy("id_item_pedido")
    .count()
    .filter(F.col("count") > 1)
    .count()
) if "id_item_pedido" in df_itp.columns else 0

# ITP-R06 — Negócio: desconto_aplicado > preco_unitario
r06 = df_itp.filter(
    F.col("desconto_aplicado") > F.col("preco_unitario")
).count() if all(c in df_itp.columns for c in ["desconto_aplicado", "preco_unitario"]) else 0

# ITP-R07 — Consistência: quantidade com casas decimais
r07 = df_itp.filter(
    F.col("quantidade") != F.floor(F.col("quantidade"))
).count() if "quantidade" in df_itp.columns else 0

# ── 4. Consolidação dos resultados ────────────────────────────────────────────
resultados_itp = [
    {"codigo": "ITP-R01", "descricao": "Nulos em colunas obrigatórias",     "tipo": "Completude",   "erro_count": r01, "total": total},
    {"codigo": "ITP-R03", "descricao": "quantidade <= 0",                    "tipo": "Domínio",      "erro_count": r03, "total": total},
    {"codigo": "ITP-R04", "descricao": "preco_unitario <= 0",                "tipo": "Domínio",      "erro_count": r04, "total": total},
    {"codigo": "ITP-R05", "descricao": "id_item_pedido duplicado no lote",   "tipo": "Unicidade",    "erro_count": r05, "total": total},
    {"codigo": "ITP-R06", "descricao": "desconto_aplicado > preco_unitario", "tipo": "Negócio",      "erro_count": r06, "total": total},
    {"codigo": "ITP-R07", "descricao": "quantidade com casas decimais",      "tipo": "Consistência", "erro_count": r07, "total": total},
]

# ── 5. Gráfico ────────────────────────────────────────────────────────────────
plot_erros(resultados_itp, "ecommerce_itens_pedido")

## 3. ecommerce_produtos
### Regras aplicadas
| Código | Tipo | Regra |

| PRD-R01 | Formato | `sku` deve ter comprimento entre 5 e 60 caracteres |

| PRD-R02 | Domínio | `preco_lista` deve ser > 0 e < 5000 |

| PRD-R03 | Completude | `is_ativo` não pode ser nulo |

| PRD-R04 | Negócio | Produto `is_ativo = true` com `preco_lista = 0` |

| PRD-R05 | Formato | `sku` não pode conter espaços ou caracteres especiais |

| PRD-R06 | Consistência | `nome_produto` não pode ser nulo ou vazio |

In [0]:
# ── 1. Leitura da Silver ──────────────────────────────────────────────────────
log.info("Lendo Silver: ecommerce_produtos")

df_prd = ler_delta_silver("ecommerce_produtos")
total  = df_prd.count()

log.info(f"Total de registros lidos: {total:,}")
log.info(f"Colunas disponíveis: {df_prd.columns}")

# ── 2. Aplicação das regras ───────────────────────────────────────────────────
from pyspark.sql import functions as F
import re

# PRD-R01 — Formato: sku fora do range 5-60 caracteres
r01 = df_prd.filter(
    (F.length(F.col("sku")) < 5) | (F.length(F.col("sku")) > 60)
).count() if "sku" in df_prd.columns else 0

# PRD-R02 — Domínio: preco_lista fora do range (0, 5000)
r02 = df_prd.filter(
    (F.col("preco_lista") <= 0) | (F.col("preco_lista") >= 5000)
).count() if "preco_lista" in df_prd.columns else 0

# PRD-R03 — Completude: is_ativo nulo
r03 = df_prd.filter(
    F.col("is_ativo").isNull()
).count() if "is_ativo" in df_prd.columns else 0

# PRD-R04 — Negócio: produto ativo com preco_lista = 0
r04 = df_prd.filter(
    (F.col("is_ativo") == True) & (F.col("preco_lista") <= 0)
).count() if all(c in df_prd.columns for c in ["is_ativo", "preco_lista"]) else 0

# PRD-R05 — Formato: sku com espaços ou caracteres especiais
r05 = df_prd.filter(
    F.col("sku").rlike(r"[^a-zA-Z0-9_\-]")
).count() if "sku" in df_prd.columns else 0

# PRD-R06 — Consistência: nome_produto nulo ou vazio
r06 = df_prd.filter(
    F.col("nome_produto").isNull() | (F.trim(F.col("nome_produto")) == "")
).count() if "nome_produto" in df_prd.columns else 0

# ── 3. Consolidação dos resultados ────────────────────────────────────────────
resultados_prd = [
    {"codigo": "PRD-R01", "descricao": "SKU fora do range 5-60 chars",       "tipo": "Formato",      "erro_count": r01, "total": total},
    {"codigo": "PRD-R02", "descricao": "preco_lista fora do range (0,5000)",  "tipo": "Domínio",      "erro_count": r02, "total": total},
    {"codigo": "PRD-R03", "descricao": "is_ativo nulo",                       "tipo": "Completude",   "erro_count": r03, "total": total},
    {"codigo": "PRD-R04", "descricao": "Produto ativo com preco_lista = 0",   "tipo": "Negócio",      "erro_count": r04, "total": total},
    {"codigo": "PRD-R05", "descricao": "SKU com caracteres inválidos",        "tipo": "Formato",      "erro_count": r05, "total": total},
    {"codigo": "PRD-R06", "descricao": "nome_produto nulo ou vazio",          "tipo": "Consistência", "erro_count": r06, "total": total},
]

# ── 4. Gráfico (eixo X forçado >= 0) ─────────────────────────────────────────
plot_erros(resultados_prd, "ecommerce_produtos")

## Resumo Consolidado — Qualidade de Dados Squad 2
### Resultado final por tabela
| Tabela | Total Registros | Regras Avaliadas | FAILs | Status |

| `ecommerce_categorias` | ~9.450 | 5 | 1 (CAT-R02) |  Atenção |

| `ecommerce_itens_pedido` | 867 | 6 | 0 |  OK |

| `ecommerce_produtos` | - | 6 | 0 |  OK |

### Principal achado
- **CAT-R02** detectou **135 registros duplicados (1.43%)** em `ecommerce_categorias`
- Causa provável: acúmulo de snapshots sem deduplicação na camada Silver
- Ação recomendada: aplicar `dropDuplicates(["id_categoria"])` no pipeline Silver

In [0]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

CORES_TABELA = {
    "ecommerce_categorias"   : "#E74C3C",
    "ecommerce_itens_pedido" : "#3498DB",
    "ecommerce_produtos"     : "#2ECC71",
}

# Consolida todos os resultados e calcula pct aqui mesmo
todos_resultados = (
    [{"tabela": "ecommerce_categorias",   **r} for r in resultados_cat] +
    [{"tabela": "ecommerce_itens_pedido", **r} for r in resultados_itp] +
    [{"tabela": "ecommerce_produtos",     **r} for r in resultados_prd]
)

df_all = pd.DataFrame(todos_resultados)
df_all["pct"] = (df_all["erro_count"] / df_all["total"] * 100).round(2)
df_all = df_all[df_all["erro_count"] > 0]

if df_all.empty:
    print("✅ Nenhum erro encontrado em nenhuma tabela!")
else:
    df_all["label"] = df_all["tabela"].str.replace("ecommerce_", "") + " › " + df_all["codigo"]
    df_all["cor"]   = df_all["tabela"].map(CORES_TABELA)
    df_all          = df_all.sort_values("erro_count", ascending=True).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(13, max(4, len(df_all) * 0.8)))

    bars = ax.barh(
        df_all["label"],
        df_all["erro_count"],
        color=df_all["cor"],
        edgecolor="white",
        height=0.6,
    )

    for bar, (_, row) in zip(bars, df_all.iterrows()):
        ax.text(
            bar.get_width() + df_all["erro_count"].max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{int(bar.get_width()):,}  ({row['pct']}%)",
            va="center", fontsize=9, color="#2C3E50",
        )

    legend_patches = [
        mpatches.Patch(color=cor, label=tabela.replace("ecommerce_", ""))
        for tabela, cor in CORES_TABELA.items()
        if tabela in df_all["tabela"].values
    ]
    ax.legend(handles=legend_patches, title="Tabela", loc="lower right", fontsize=8)

    ax.set_title("Consolidado de Erros — Squad 2", fontsize=13, fontweight="bold", pad=14)
    ax.set_xlabel("Quantidade de registros com erro", fontsize=10)
    max_val = df_all["erro_count"].max()
    ax.set_xlim(0, max_val * 1.18 if max_val > 0 else 10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()